### IMPORT & ĐỌC DỮ LIỆU

In [2]:
import pandas as pd

menu = pd.read_csv("../data_clean/menu_clean.csv")
order = pd.read_csv("../data_clean/order_log_clean.csv")
cost = pd.read_csv("../data_clean/cost_raw_material_clean.csv")

print("MENU:")
display(menu.head())

print("ORDER:")
display(order.head())

print("COST:")
display(cost.head())

display(cost.head())

MENU:


,product_id,product_name,price
0,CF01,Ca Phe Den,45000.0
1,CF02,Ca Phe Sua,35000.0
2,CF03,Bac Xiu,35000.0
3,CF04,Tra Dao,45000.0
4,CF05,Sinh To Xoai,30.0


ORDER:


,order_id,product_id,order_time,qty
0,OD2002,CF03,2024-01-01 14:12:00,2.0
1,OD2003,CF03,2024-01-01 06:16:00,2.0
2,OD2004,CF02,2024-01-01 04:55:00,1.0
3,OD2006,CF02,2024-01-01 06:56:00,3.0
4,OD2009,CF06,NaN,1.0


COST:


,product_id,unit_cost
0,CF01,8000.0
1,CF02,15000.0
2,CF03,NaN
3,CF04,9000.0
4,CF05,15000.0


,product_id,unit_cost
0,CF01,8000.0
1,CF02,15000.0
2,CF03,NaN
3,CF04,9000.0
4,CF05,15000.0


### MERGE MENU VỚI ORDER

In [3]:
order_menu = order.merge(
    menu,
    on="product_id",
    how="left",
    indicator=True
)

print("KẾT QUẢ MERGE ORDER + MENU:")
display(order_menu.head())

print("THỐNG KÊ TRẠNG THÁI MERGE:")
print(order_menu["_merge"].value_counts())


KẾT QUẢ MERGE ORDER + MENU:


,order_id,product_id,order_time,qty,product_name,price,_merge
0,OD2002,CF03,2024-01-01 14:12:00,2.0,Bac Xiu,35000.0,both
1,OD2003,CF03,2024-01-01 06:16:00,2.0,Bac Xiu,35000.0,both
2,OD2004,CF02,2024-01-01 04:55:00,1.0,Ca Phe Sua,35000.0,both
3,OD2006,CF02,2024-01-01 06:56:00,3.0,Ca Phe Sua,35000.0,both
4,OD2009,CF06,NaN,1.0,Matcha Latte,35000.0,both


THỐNG KÊ TRẠNG THÁI MERGE:
_merge
both          63
left_only      0
right_only     0
Name: count, dtype: int64


### MERGE THÊM VỚI COST

In [4]:
order_full = order_menu.merge(
    cost,
    on="product_id",
    how="left",
    indicator="cost_merge"
)

print("DỮ LIỆU SAU KHI MERGE THÊM COST:")
display(order_full.head())

print("THỐNG KÊ TRẠNG THÁI MERGE COST:")
print(order_full["cost_merge"].value_counts())


DỮ LIỆU SAU KHI MERGE THÊM COST:


,order_id,product_id,order_time,qty,product_name,price,_merge,unit_cost,cost_merge
0,OD2002,CF03,2024-01-01 14:12:00,2.0,Bac Xiu,35000.0,both,NaN,both
1,OD2003,CF03,2024-01-01 06:16:00,2.0,Bac Xiu,35000.0,both,NaN,both
2,OD2004,CF02,2024-01-01 04:55:00,1.0,Ca Phe Sua,35000.0,both,15000.0,both
3,OD2006,CF02,2024-01-01 06:56:00,3.0,Ca Phe Sua,35000.0,both,15000.0,both
4,OD2009,CF06,NaN,1.0,Matcha Latte,35000.0,both,9000.0,both


THỐNG KÊ TRẠNG THÁI MERGE COST:
cost_merge
both          63
left_only      0
right_only     0
Name: count, dtype: int64


### PHÁT HIỆN MISMACTH HOẶC MÓN KHÔNG CÓ COST

In [5]:
order_missing_menu = order_menu[order_menu["_merge"] == "left_only"]

print("CÁC ORDER KHÔNG CÓ THÔNG TIN MENU:")
display(order_missing_menu)

CÁC ORDER KHÔNG CÓ THÔNG TIN MENU:


,order_id,product_id,order_time,qty,product_name,price,_merge


#### PHÁT HIỆN MÓN KHÔNG CÓ CHI PHÍ NGUYÊN LIỆU

In [6]:
missing_cost_products = order_full[order_full["unit_cost"].isna()]

print("CÁC MÓN KHÔNG CÓ THÔNG TIN CHI PHÍ:")
display(missing_cost_products)


CÁC MÓN KHÔNG CÓ THÔNG TIN CHI PHÍ:


,order_id,product_id,order_time,qty,product_name,price,_merge,unit_cost,cost_merge
0,OD2002,CF03,2024-01-01 14:12:00,2.0,Bac Xiu,35000.0,both,NaN,both
1,OD2003,CF03,2024-01-01 06:16:00,2.0,Bac Xiu,35000.0,both,NaN,both
30,OD2043,CF03,NaN,3.0,Bac Xiu,35000.0,both,NaN,both
34,OD2053,CF03,NaN,3.0,Bac Xiu,35000.0,both,NaN,both
49,OD2076,CF03,2024-01-01 02:21:00,3.0,Bac Xiu,35000.0,both,NaN,both
55,OD2086,CF03,NaN,3.0,Bac Xiu,35000.0,both,NaN,both
57,OD2089,CF03,2024-01-01 07:03:00,3.0,Bac Xiu,35000.0,both,NaN,both
60,OD2092,CF03,NaN,3.0,Bac Xiu,35000.0,both,NaN,both


#### TÍNH DOANH THU & LỢI NHUẬN

In [7]:
order_full["revenue"] = order_full["qty"] * order_full["price"]
order_full["profit"] = order_full["price"] - order_full["unit_cost"]

print("DỮ LIỆU SAU KHI TÍNH DOANH THU & LỢI NHUẬN:")
display(order_full.head())


DỮ LIỆU SAU KHI TÍNH DOANH THU & LỢI NHUẬN:


,order_id,product_id,order_time,qty,product_name,price,_merge,unit_cost,cost_merge,revenue,profit
0,OD2002,CF03,2024-01-01 14:12:00,2.0,Bac Xiu,35000.0,both,NaN,both,70000.0,NaN
1,OD2003,CF03,2024-01-01 06:16:00,2.0,Bac Xiu,35000.0,both,NaN,both,70000.0,NaN
2,OD2004,CF02,2024-01-01 04:55:00,1.0,Ca Phe Sua,35000.0,both,15000.0,both,35000.0,20000.0
3,OD2006,CF02,2024-01-01 06:56:00,3.0,Ca Phe Sua,35000.0,both,15000.0,both,105000.0,20000.0
4,OD2009,CF06,NaN,1.0,Matcha Latte,35000.0,both,9000.0,both,35000.0,26000.0


#### HOÀN THIỆN DATASET MERGE

In [8]:
final_data = order_full.drop(columns=["_merge", "cost_merge"])

print("DATASET HOÀN CHỈNH DÙNG CHO PHÂN TÍCH:")
display(final_data.head())


DATASET HOÀN CHỈNH DÙNG CHO PHÂN TÍCH:


,order_id,product_id,order_time,qty,product_name,price,unit_cost,revenue,profit
0,OD2002,CF03,2024-01-01 14:12:00,2.0,Bac Xiu,35000.0,NaN,70000.0,NaN
1,OD2003,CF03,2024-01-01 06:16:00,2.0,Bac Xiu,35000.0,NaN,70000.0,NaN
2,OD2004,CF02,2024-01-01 04:55:00,1.0,Ca Phe Sua,35000.0,15000.0,35000.0,20000.0
3,OD2006,CF02,2024-01-01 06:56:00,3.0,Ca Phe Sua,35000.0,15000.0,105000.0,20000.0
4,OD2009,CF06,NaN,1.0,Matcha Latte,35000.0,9000.0,35000.0,26000.0


#### XUẤT FILE DỮ LIỆU MERGE

In [10]:
import os
os.makedirs("../data_merge", exist_ok=True)

final_data.to_csv(
    "../data_merge/order_full_merged.csv",
    index=False
)

print("ĐÃ XUẤT FILE order_full_merged.csv")

ĐÃ XUẤT FILE order_full_merged.csv
